# Environment / Dependencies Setup
The cell below installs all required Python packages:

In [ ]:
!pip install -qU qdrant-client fastembed langchain-openai langchain-text-splitters pypdf requests python-dotenv ipython

### Step 1 — Imports

In [ ]:
import hashlib
import os
import uuid

import requests
from dotenv import load_dotenv
from fastembed import LateInteractionTextEmbedding
from IPython.display import Markdown, display
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from qdrant_client import QdrantClient, models

### Step 2 — Configure Models

In [ ]:
# Load the secrets (QDRANT_URL, QDRANT_API_KEY, OPENROUTER_API_KEY) from .env
load_dotenv(".env")

# Initialize Chat Model
llm = ChatOpenAI(
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    # openai-compatible client, but routed to OpenRouter's servers
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="nvidia/nemotron-3-super-120b-a12b:free",
    temperature=0.0
)


In [ ]:
# ColBERT embeds every TOKEN, so the output is a (num_tokens, 128) matrix — not a single vector
colbert = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

### Step 3 — Initialize Qdrant with Multivector Support or Use Existing Collection if Already Made

In [ ]:
client = QdrantClient(
    url=os.getenv("QDRANT_URL"), 
    api_key=os.getenv("QDRANT_API_KEY"),
    timeout=300,
)
COLLECTION_NAME = "colbert_late_interaction"

# FIRST TIME RUN: creates a fresh collection (wipes & rebuilds it), then data is uploaded in Step 8
# ColBERT stores token MATRICES, so Qdrant needs a MULTIVECTOR collection
# compared with MaxSim instead of plain cosine similarity
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=128,
        distance=models.Distance.COSINE,
        multivector_config=models.MultiVectorConfig(
            comparator=models.MultiVectorComparator.MAX_SIM
        )
    )
)

In [ ]:
# ALREADY RAN THIS NOTEBOOK? Uncomment this cell if you have already
# made the DB (collection) / uploaded data, then run it instead of Cell A
# (Cell A wipes and recreates the collection, so only use it the first time).
# client = QdrantClient(
#     url=os.getenv("QDRANT_URL"), 
#     api_key=os.getenv("QDRANT_API_KEY"),
#     timeout=300,
# )
# COLLECTION_NAME = "colbert_late_interaction"
# No recreate_collection here — the collection already exists.

### Step 4 — Download & Extract Document

In [ ]:
# BERT paper (source document for this lab)
PDF_URL = "https://arxiv.org/pdf/1810.04805.pdf"
PDF_FILENAME = "bert_paper.pdf"

# Download the PDF and save it locally; raise an error on HTTP failure
response = requests.get(PDF_URL)
response.raise_for_status()

with open(PDF_FILENAME, "wb") as f:
    f.write(response.content)

In [ ]:
# Extract text from every page and join pages with newlines into one string
reader = PdfReader(PDF_FILENAME)
raw_text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])

print(f"Extracted {len(raw_text)} characters.")

### Step 5 — Chunk the Document

In [ ]:
# Smaller chunks than Lab 3: ColBERT embeds every token, so big chunks would mean huge matrices
chunk_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = chunk_splitter.split_text(raw_text)

print(f"Created {len(chunks)} chunks.")

### Step 6 — Generate Stable Chunk IDs
Stable IDs prevent duplicate Qdrant points on re-runs.

In [ ]:
def stable_id(text):
    """Deterministic ID derived from the chunk text, so the same chunk always gets the same ID across runs."""
    return str(uuid.uuid5(uuid.NAMESPACE_URL, text))


chunk_ids = [stable_id(chunk) for chunk in chunks]

### Step 7 — Generate ColBERT Multi-Vector Embeddings

In [ ]:
# Each chunk becomes a token matrix of shape (num_tokens, 128), not a single vector
chunk_embeddings = list(colbert.embed(chunks))

# Teaching sanity check: expect (num_tokens, 128)
print(f"First chunk embedding shape: {chunk_embeddings[0].shape}")

### Step 8 — Ingest Matrices into Qdrant

In [ ]:
# one qdrant point per chunk: stable id + token matrix + raw text payload
points = [
    models.PointStruct(
        # stable id means re-running this overwrites a chunk instead of duplicating it
        id=stable_id(chunk),
        vector=embedding.tolist(),
        payload={"text": chunk},
    )
    for chunk, embedding in zip(chunks, chunk_embeddings)
]

# Upload in small batches (multivector points are big; large batches time out on free tier)
client.upload_points(collection_name=COLLECTION_NAME, points=points, batch_size=10)

print(f"Uploaded {len(points)} ColBERT points.")

### Step 9 — Define the Late Interaction Query Function

In [ ]:
def retrieve(query, top_k=3):
    """Embed the query as a token matrix and run a MaxSim (late interaction) search."""
    # query_embed yields a generator; [0] takes the single query token matrix
    query_matrix = list(colbert.query_embed(query))[0]
    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_matrix.tolist(),
        limit=top_k,
    ).points
    return hits

### Step 10 — Define the RAG Generation Chain

In [ ]:
qa_template = """
You are a technical assistant. Answer the question using ONLY the provided context.

Context:
{context}

Question: {question}

Respond in exactly this format:

### Final Answer
<a clear, direct answer to the question>

### AI Tracing & Explainability
<Explain step by step how you arrived at the answer. Refer to the sources you used by their label (e.g. "Source 1"), state what each one contributed, and why it was relevant. Do not copy large blocks of text from the context — explain in your own words. Use as many lines as you need.>
"""

qa_prompt = ChatPromptTemplate.from_template(qa_template)

Each retrieved hit gets a label carrying its MaxSim score, so the LLM can cite a source and the reader can see how well it matched.

In [ ]:
def format_hits(hits):
    """Label each retrieved chunk with its MaxSim score and payload text."""
    formatted = []
    for i, hit in enumerate(hits):
        formatted.append(f"[Source {i+1}] (MaxSim score: {hit.score})\n{hit.payload['text']}")
    return "\n\n".join(formatted)

# Prompt → LLM → plain string output
generation_chain = qa_prompt | llm | StrOutputParser()

### Step 11 — Full Pipeline

In [ ]:
def run_colbert_rag(query, top_k=3):
    """
    Runs the full pipeline: late-interaction retrieval, formatting, LLM generation.
    Displays the answer and returns both the hits and the answer.
    """
    hits = retrieve(query, top_k=top_k)
    context = format_hits(hits)
    answer = generation_chain.invoke({"context": context, "question": query})
    display(Markdown(answer))
    return hits, answer

### Step 12 — Execute Query

In [ ]:
query = "What pre-training tasks does BERT use to learn bidirectional representations?"
hits, answer = run_colbert_rag(query)